# Solution F1 · Scaling and metric: what really matters, on three datasets

**Type**: Compare

## Objective

Systematically compare three treatments of the scales (none, `StandardScaler`, `MinMaxScaler`) and four metrics (Euclidean, Manhattan, Chebyshev, cosine) on three datasets (wines, breast tumors, handwritten digits) with
a repeated cross-validation, and establish **which choice weighs most** in each case.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_breast_cancer, load_digits, load_wine
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler

datasets = {"wine": load_wine(return_X_y=True), "cancer": load_breast_cancer(return_X_y=True), "digits": load_digits(return_X_y=True)}
scalers = {"none": None, "standard": StandardScaler, "minmax": MinMaxScaler}
metrics = ["euclidean", "manhattan", "chebyshev", "cosine"]
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=0)

## Commented solution

In [ ]:
index = pd.MultiIndex.from_product([datasets, scalers], names=["dataset", "scaling"])
table = pd.DataFrame(index=index, columns=metrics, dtype=float)
for (name, sc) in table.index:
    X, y = datasets[name]
    steps = [] if sc == "none" else [scalers[sc]()]
    for metric in metrics:
        model = make_pipeline(*steps, KNeighborsClassifier(7, algorithm="brute", metric=metric))
        table.loc[(name, sc), metric] = cross_val_score(model, X, y, cv=cv).mean()
table.round(3)

In [ ]:
best, scaling_gain, metric_spread, dominant_factor = {}, {}, {}, {}
for name in datasets:
    t = table.loc[name]
    best[name] = t.stack().idxmax()
    scaling_gain[name] = t.loc[["standard", "minmax"]].to_numpy().max() - t.loc["none"].max()
    metric_spread[name] = t.loc["standard"].max() - t.loc["standard"].min()
    dominant_factor[name] = "scaling" if scaling_gain[name] > metric_spread[name] else "metric"
print(pd.DataFrame({"best": best, "scaling gain": scaling_gain, "metric spread": metric_spread, "dominant": dominant_factor}).round(3))

## Check

In [ ]:
with verify():
    check(table.shape == (9, 4) and table.notna().all().all() and list(table.columns) == metrics, "9 rows (3 datasets x 3 scalings) and 4 metrics, with no gaps", "one accuracy for every combination")
    check(table.loc[("wine", "none")].max() < 0.82 and table.loc[("wine", "standard")].min() > 0.92, "wine: without scaling below 0.82; with StandardScaler above 0.92 with any metric", "the scales of the wines are very different")
    check(scaling_gain["wine"] > 0.15 and 0.005 < scaling_gain["cancer"] < 0.08, "gain of scaling: over 0.15 on the wines, between 0.005 and 0.08 on the cancer data", "best with scaler minus best without")
    check(scaling_gain["digits"] < 0.005 and table.loc[("digits", "standard"), "euclidean"] < table.loc[("digits", "none"), "euclidean"], "on the digits scaling does not help (gain below 0.005) and StandardScaler worsens the Euclidean", "the pixels already have the same scale")
    check(best["wine"][0] in ("standard", "minmax") and best["digits"][0] in ("none", "minmax"), "the best treatment: a scaler on the wines; none (or minmax) on the digits", "idxmax of the table")
    check(metric_spread["wine"] < 0.06 and scaling_gain["wine"] > 4 * metric_spread["wine"], "on the wines the metric changes less than 0.06 and scaling weighs over 4 times more", "spread = max - min across metrics with StandardScaler")
    check(dominant_factor["wine"] == "scaling" and dominant_factor["digits"] == "metric", "the dominant factor: scaling for the wines, metric for the digits (on the cancer data the two weigh about the same)", "compare scaling_gain and metric_spread")
done()

## What you learned

- **On the wines scaling is everything**: without a scaler the accuracy sits between 0.67 and 0.80 (`proline` decides alone), with `StandardScaler` or `MinMaxScaler` it goes up to 0.94-0.97 with any metric. Changing the metric moves it by 3 points, changing the scaling by 27.
- **On the cancer data scaling helps only a little** (about 4 points) because the most variable features (areas) are also informative; the metric counts almost as much (2-3 points).
- **On the digits scaling is not needed and `StandardScaler` makes it worse** (0.972 against 0.983): the 64 pixels are already on the same scale (0-16), and standardizing them magnifies the small variations of the almost always dark pixels, which are noise. Here `minmax`
  (a constant factor) coincides with not scaling, and the cosine does not get worse with it (0.984 both with and without scaling). **The rule is not "always scale": it is "bring the columns to a comparable weight for the problem".**
- Manhattan is often a bit better than Euclidean with many features and outliers; Chebyshev (only the maximum gap) is almost always worse; the cosine works well where the direction matters (digits), badly on unscaled wines... and even better after standardization.
  The metric is a second-order hyperparameter: scales first, then the rest.

## Common pitfall

Applying `StandardScaler` to everything out of habit. With already homogeneous data (pixels, counts, percentages) it can make things worse, and the signal is not an error but a lower number that nobody compares with the baseline without scaling.

## Interview question

> *Click on a question to show its answer.*

<details><summary>Do I always have to scale the features before a KNN?</summary>

Almost always, because the distance sums the differences and different scales decide by themselves. But it is not a rule: if the features already have the same meaning and the same scale (pixels, histograms, normalized counts) standardizing can amplify the noise
of the nearly constant columns. I always compare with and without a scaler in cross-validation and also look at the metric, which usually counts less.

</details>